# **Preprocessing Datasets** 

### **STEP 1.1 — Convert old .doc files to modern .docx files**

Why: python-docx cannot read old binary .doc files.
     LibreOffice does the conversion reliably and preserves structure.
 
Input:

    data/raw/doc/             || Original .doc files

Output:

    data/processed/docx/      || Converted .docx files

    outputs/logs/             || Conversion reports

In [5]:
import json
import hashlib
import subprocess
import time
from pathlib import Path
import re
import pandas as pd
from docx.text.paragraph import Paragraph
from collections import defaultdict
from pathlib import Path
from docx import Document
from docx.oxml.ns import qn
from docx.text.paragraph import Paragraph


In [6]:
%pip install python-docx

Note: you may need to restart the kernel to use updated packages.


In [7]:
import os, subprocess

os.environ["PATH"] += ":/Applications/LibreOffice.app/Contents/MacOS"  # add to PATH

# on Mac, LibreOffice binary is called "soffice" not "libreoffice"
result = subprocess.run(["which", "soffice"], capture_output=True, text=True)
print("LibreOffice found at:", result.stdout.strip() or "NOT FOUND")

# quick version check to confirm it works
version = subprocess.run(["soffice", "--version"], capture_output=True, text=True)
print("Version:", version.stdout.strip())

LibreOffice found at: /Applications/LibreOffice.app/Contents/MacOS/soffice
Version: LibreOffice 26.2.5.2 cd7284b4cbbfeb507e630c1aac019f4157393acb


In [8]:
# Folder configuration 

# Main folder of the ESG RAG project
PROJECT_ROOT = Path("/Users/tanggiee/Desktop/RAG_AI/esg_rag_project")

# Original legacy .doc documents
RAW_DOC_FOLDER = Path("/Users/tanggiee/Desktop/RAG_AI/Documents")

# Documents converted from .doc to .docx
CONVERTED_DOCX_FOLDER = PROJECT_ROOT / "data" / "processed" / "docx"

# Cleaned document text prepared for chunking
CLEANED_TEXT_FOLDER = PROJECT_ROOT / "data" / "processed" / "cleaned"

# Processing reports and error logs
LOG_FOLDER = PROJECT_ROOT / "outputs" / "logs"

# Create folders that will receive processing outputs
CONVERTED_DOCX_FOLDER.mkdir(parents=True, exist_ok=True)
CLEANED_TEXT_FOLDER.mkdir(parents=True, exist_ok=True)
LOG_FOLDER.mkdir(parents=True, exist_ok=True)

In [9]:
# create required folders 
# Create the output folders if they do not already exist
CONVERTED_DOCX_FOLDER.mkdir(parents=True, exist_ok=True)
LOG_FOLDER.mkdir(parents=True, exist_ok=True)

In [10]:
# check folders 
print(f"Original .doc files: {RAW_DOC_FOLDER.resolve()}")
print(f"Converted .docx files: {CONVERTED_DOCX_FOLDER.resolve()}")
print(f"Processing logs: {LOG_FOLDER.resolve()}")


Original .doc files: /Users/tanggiee/Desktop/RAG_AI/Documents
Converted .docx files: /Users/tanggiee/Desktop/RAG_AI/esg_rag_project/data/processed/docx
Processing logs: /Users/tanggiee/Desktop/RAG_AI/esg_rag_project/outputs/logs


**Steps**: 
- convert_single_file() converts one .doc file into .docx.
- convert_all_docs() finds and converts all .doc files in a folder, skips completed files, and saves a JSON report.

In [11]:
# Convert one legacy .doc file to .docx using LibreOffice.

def convert_single_file(doc_path: Path, output_folder: Path) -> dict:
    start_time = time.time()
    output_path = output_folder / f"{doc_path.stem}.docx"

    # Default result; values will be updated after the conversion attempt
    result = {
        "file": doc_path.name,
        "status": "failed",
        "output_path": None,
        "error": None,
        "duration_sec": None,}

    try:
        # Run LibreOffice without opening its graphical interface
        process = subprocess.run(
            [
                "soffice",
                "--headless",
                "--convert-to", "docx",
                "--outdir", str(output_folder),
                str(doc_path),
            ],
            capture_output=True,  # Capture messages from LibreOffice
            text=True,            # Return messages as normal strings
            timeout=120,          # Stop if conversion takes over 2 minutes
        )

        # A successful conversion should create the expected .docx file
        if process.returncode == 0 and output_path.exists():
            result["status"] = "success"
            result["output_path"] = str(output_path)
        else:
            result["error"] = (
                process.stderr.strip()
                or process.stdout.strip()
                or "LibreOffice did not create the expected .docx file."
            )

    except subprocess.TimeoutExpired:
        result["status"] = "timeout"
        result["error"] = "Conversion exceeded the 120-second time limit."

    except FileNotFoundError:
        result["status"] = "error"
        result["error"] = "LibreOffice is not installed or cannot be found."

    except Exception as error:
        result["status"] = "error"
        result["error"] = str(error)

    # Record the total processing time
    result["duration_sec"] = round(time.time() - start_time, 2)

    return result

In [12]:
# Convert every .doc file in a folder and save a JSON report.
def convert_all_docs(raw_folder: Path, output_folder: Path, log_folder: Path,) -> dict:

    # Make sure the required folders exist
    output_folder.mkdir(parents=True, exist_ok=True)
    log_folder.mkdir(parents=True, exist_ok=True)

    # Find all .doc files and arrange them alphabetically by filename
    doc_files = sorted(raw_folder.glob("*.doc"))

    if not doc_files:
        print(f"No .doc files found in: {raw_folder.resolve()}")
        return {}

    print(f"Found {len(doc_files)} .doc files.\n")

    results = []

    for number, doc_path in enumerate(doc_files, start=1):
        output_path = output_folder / f"{doc_path.stem}.docx"

        # Do not repeat conversions that were already completed
        if output_path.exists():
            result = {
                "file": doc_path.name,
                "status": "skipped",
                "output_path": str(output_path),
                "error": None,
                "duration_sec": 0,
            }
        else:
            result = convert_single_file(doc_path, output_folder)

        results.append(result)

        # Display progress for the current file
        print(
            f"[{number}/{len(doc_files)}] "
            f"{result['status'].upper():8} | "
            f"{result['duration_sec']:6.2f}s | "
            f"{doc_path.name}"
        )

        if result["error"]:
            print(f"    Error: {result['error']}")

    # Count each result type separately
    summary = {
        "total": len(doc_files),
        "success": sum(r["status"] == "success" for r in results),
        "skipped": sum(r["status"] == "skipped" for r in results),
        "failed": sum(
            r["status"] in {"failed", "timeout", "error"}
            for r in results
        ),
    }

    report = {
        "summary": summary,
        "files": results,}
    # Save the complete report as a readable JSON file
    report_path = log_folder / "step_01_doc_to_docx_report.json"

    with report_path.open("w", encoding="utf-8") as file:
        json.dump(report, file, indent=2, ensure_ascii=False)

    # Display the final conversion summary
    print("\n" + "=" * 50)
    print(f"Total:      {summary['total']}")
    print(f"Converted:  {summary['success']}")
    print(f"Skipped:    {summary['skipped']}")
    print(f"Failed:     {summary['failed']}")
    print(f"Report:     {report_path}")
    print("=" * 50)

    return report

In [13]:
# quick check on the number of documents 
from pathlib import Path

PROJECT_ROOT = Path("/Users/tanggiee/Desktop/RAG_AI/esg_rag_project")  # absolute path

doc_files  = list((PROJECT_ROOT / "data" / "raw" / "doc").glob("*.doc"))   # original .doc files
docx_files = list((PROJECT_ROOT / "data" / "processed" / "docx").glob("*.docx"))  # converted
txt_files  = list((PROJECT_ROOT / "data" / "processed" / "cleaned").glob("*.txt"))  # cleaned

print(f"Raw      .doc  : {len(doc_files)}")   # should be 600
print(f"Converted .docx: {len(docx_files)}")  # populated after step 1.1
print(f"Cleaned   .txt : {len(txt_files)}")   # populated after step 1.3

Raw      .doc  : 0
Converted .docx: 407
Cleaned   .txt : 406


### **STEP 1.2 — Extract Text, Metadata & ESG Classification**

Input  : data/processed/docx/

Output : data/processed/cleaned/   (one .txt per doc)

outputs/logs/corpus_registry.json

##### **a. Clean data**

In [14]:
# Find the repeated issuing bodies and remove boilerplates

# Normalize spacing so visually identical lines can be counted together
def normalize_line(line: str) -> str:
    return " ".join(line.split()).strip()


# Find repeated header/footer lines and possible issuing bodies.
def inspect_document_headers(docx_folder: Path, header_lines: int = 20, footer_lines: int = 5) -> tuple:

    docx_files = sorted(docx_folder.rglob("*.docx"))

    # Store each line or issuing body with the documents where it appears
    repeated_line_sources = defaultdict(set)
    issuing_body_sources = defaultdict(set)

    # Find authority names before words such as "promulgates"
    issuing_body_pattern = re.compile(r"(The Government|The Prime Minister|"
                                      r"Minister of [A-Za-z ,&-]+?)\s+promulgates?", 
                                      flags=re.IGNORECASE)

    for docx_path in docx_files:
        try:
            document = Document(docx_path)

            # Extract and normalize non-empty paragraphs
            lines = [normalize_line(paragraph.text)
                     for paragraph in document.paragraphs
                     if normalize_line(paragraph.text)]

            # Inspect only likely header and footer areas
            inspected_lines = lines[:header_lines] + lines[-footer_lines:]

            # Count a repeated line only once per document
            for line in set(inspected_lines):
                repeated_line_sources[line].add(docx_path.name)

            # Search the beginning of the document for the issuing authority
            header_text = "\n".join(lines[:header_lines])

            for match in issuing_body_pattern.finditer(header_text):
                issuing_body = normalize_line(match.group(1)).title()
                issuing_body_sources[issuing_body].add(docx_path.name)

        except Exception as error:
            print(f"Could not read {docx_path.name}: {error}")

    # Convert repeated-line results into a table
    repeated_lines = pd.DataFrame([
        {
            "line": line,
            "document_count": len(filenames),
            "example_files": sorted(filenames)[:5]
        }
        for line, filenames in repeated_line_sources.items()
    ])

    # Convert issuing-body results into a table
    issuing_bodies = pd.DataFrame([
        {
            "issuing_body": issuing_body,
            "document_count": len(filenames),
            "example_files": sorted(filenames)[:5]
        }
        for issuing_body, filenames in issuing_body_sources.items()
    ])

    # Show the most frequently repeated results first
    if not repeated_lines.empty:
        repeated_lines = repeated_lines.sort_values("document_count",
                                                    ascending=False).reset_index(drop=True)

    if not issuing_bodies.empty:
        issuing_bodies = issuing_bodies.sort_values("document_count",
                                                    ascending=False).reset_index(drop=True)

    return repeated_lines, issuing_bodies

In [16]:
# Load saved header inspection or run it when no checkpoint exists
REPEATED_LINES_PATH = LOG_FOLDER / "repeated_lines.json"
ISSUING_BODIES_PATH = LOG_FOLDER / "issuing_bodies.json"
FORCE_HEADER_INSPECTION = False

if REPEATED_LINES_PATH.exists() and ISSUING_BODIES_PATH.exists() and not FORCE_HEADER_INSPECTION:
    repeated_lines = pd.read_json(REPEATED_LINES_PATH)
    issuing_bodies = pd.read_json(ISSUING_BODIES_PATH)
    print(f"Loaded saved inspection: {len(repeated_lines)} repeated lines, {len(issuing_bodies)} issuing bodies.")
else:
    repeated_lines, issuing_bodies = inspect_document_headers(
        docx_folder=CONVERTED_DOCX_FOLDER, header_lines=20, footer_lines=5)
    repeated_lines.to_json(REPEATED_LINES_PATH, orient="records", indent=2, force_ascii=False)
    issuing_bodies.to_json(ISSUING_BODIES_PATH, orient="records", indent=2, force_ascii=False)
    print("Header inspection completed and saved.")

Header inspection completed and saved.


In [ ]:
# Show lines found in at least three documents
# common_lines = repeated_lines[repeated_lines["document_count"] >= 10]

# display(common_lines)
# display(issuing_bodies)

In [17]:
# Return the unique issuing bodies found by the current extraction
issuing_body_list = (issuing_bodies["issuing_body"]
                     .dropna()
                     .drop_duplicates()
                     .tolist())

# Display one issuing body per line
for issuing_body in issuing_body_list:
    print(issuing_body)

The Government
Minister Of Agriculture And Environment
Minister Of Agriculture And Rural Development
Minister Of Finance
Minister Of Industry And Trade
Minister Of Natural Resources And Environment Hereby
The Prime Minister
Minister Of Health
Minister Of Agriculture And Rural Development Hereby
Minister Of Natural Resources And Environment
Minister Of Finance Of Vietnam
Minister Of Agriculture And Environment Of Vietnam Hereby
Minister Of Construction
Minister Of Agriculture And Environment Hereby
Minister Of Health Hereby
Minister Of Industry And Trade Hereby
Minister Of Science And Technology
Minister Of Agriculture And Environment Of Vietnam
Minister Of Natural Resources And Environment Of Vietnam
Minister Of Labor - War Invalids And Social Affairs
Minister Of Natural Resources And Environment Of Vietnam Hereby
Minister Of Labour, War Invalids And Social Affairs
Minister Of Transport Hereby
Minister Of Labor, War Invalids And Social Affairs
Minister Of Agriculture And Rural Developm

In [18]:
# define document types 

# Possible document-type headings found in Vietnamese legal documents
DOCUMENT_TYPE_LABELS = {
    "CONSTITUTION",
    "CODE",
    "LAW",
    "RESOLUTION",
    "JOINT RESOLUTION",
    "ORDINANCE",
    "ORDER",
    "DECREE",
    "DECISION",
    "CIRCULAR",
    "JOINT CIRCULAR",
    "DIRECTIVE",
    "NOTIFICATION"
}


def normalize_heading(text: str) -> str:
    """Remove extra spaces and standardize text for comparison."""

    return " ".join(text.upper().split()).strip(" .:-")


def identify_document_types(docx_folder: Path,
                            search_lines: int = 20) -> tuple:
    """Identify how document types are written across all .docx files."""

    # Find all converted Word documents
    docx_files = sorted(docx_folder.rglob("*.docx"))
    document_results = []

    for docx_path in docx_files:
        try:
            document = Document(docx_path)

            # Read the first non-empty paragraphs where the type normally appears
            header_lines = [paragraph.text.strip()
                            for paragraph in document.paragraphs[:search_lines]
                            if paragraph.text.strip()]

            observed_type = "UNKNOWN"
            observed_line = None

            # Find an exact document-type heading such as DECREE or CIRCULAR
            for line in header_lines:
                normalized_line = normalize_heading(line)

                if normalized_line in DOCUMENT_TYPE_LABELS:
                    observed_type = normalized_line.title()
                    observed_line = line
                    break

            # Extract the identifying portion of the filename
            filename_code = docx_path.stem.split("_m_")[0]

            document_results.append({
                "filename": docx_path.name,
                "filename_code": filename_code,
                "observed_type": observed_type,
                "observed_line": observed_line
            })

        except Exception as error:
            document_results.append({
                "filename": docx_path.name,
                "filename_code": None,
                "observed_type": "READ ERROR",
                "observed_line": str(error)
            })

    # One row for every document
    document_types = pd.DataFrame(document_results)

    # Summary of the document-type labels found
    type_summary = (document_types.groupby("observed_type", dropna=False)
                    .agg(document_count=("filename", "count"),
                         example_files=("filename",
                                        lambda files: list(files[:5])))
                    .reset_index()
                    .sort_values("document_count", ascending=False)
                    .reset_index(drop=True))

    return type_summary, document_types

In [19]:
# Identify document-type labels across the complete corpus
DOCUMENT_TYPES_PATH = LOG_FOLDER / "document_types.json"
TYPE_SUMMARY_PATH = LOG_FOLDER / "document_type_summary.json"
FORCE_TYPE_INSPECTION = False

if DOCUMENT_TYPES_PATH.exists() and TYPE_SUMMARY_PATH.exists() and not FORCE_TYPE_INSPECTION:
    document_types = pd.read_json(DOCUMENT_TYPES_PATH)
    type_summary = pd.read_json(TYPE_SUMMARY_PATH)
    print(f"Loaded saved document types: {len(document_types)} documents.")
else:
    type_summary, document_types = identify_document_types(
        docx_folder=CONVERTED_DOCX_FOLDER, search_lines=20)
    document_types.to_json(DOCUMENT_TYPES_PATH, orient="records", indent=2, force_ascii=False)
    type_summary.to_json(TYPE_SUMMARY_PATH, orient="records", indent=2, force_ascii=False)
    print("Document-type inspection completed and saved.")

Document-type inspection completed and saved.


In [20]:
# Show how document types are normally written
display(type_summary)

,observed_type,document_count,example_files
0,Circular,158,"[01_2021_TT-BXD_m_488637.docx, 01_2022_TT-BNNP..."
1,Decree,102,"[01_2019_ND-CP_m_420475.docx, 02_VBHN-BKHCN_m_..."
2,Law,64,"[02_2026_QH16_m_706740.docx, 04_2017_QH14_m_35..."
3,Decision,56,"[01_2022_QD-TTg_m_567258.docx, 04_2017_QD-TTg_..."
4,UNKNOWN,13,"[101_VBHN-VPQH_m_694902.docx, 11_VBHN-BTC_m_60..."
5,Resolution,10,"[138_NQ-CP_m_544532.docx, 189_2025_QH15_m_6536..."
6,Directive,1,[03_CT-NHNN_m_348086.docx]
7,Notification,1,[30_TB-VPCP_m_507657.docx]
8,Order,1,[11_2001_L-CTN_m_71892.docx]
9,Ordinance,1,[10_2014_UBTVQH13_m_267874.docx]


In [21]:
# Display documents whose type could not be found
unknown_document_list = document_types[document_types["observed_type"] == "UNKNOWN"]

display(unknown_document_list)

,filename,filename_code,observed_type,observed_line
67,101_VBHN-VPQH_m_694902.docx,101_VBHN-VPQH,UNKNOWN,NaN
101,11_VBHN-BTC_m_609280.docx,11_VBHN-BTC,UNKNOWN,NaN
105,125_VBHN-VPQH_m_682303.docx,125_VBHN-VPQH,UNKNOWN,NaN
154,16_2023_QH15_m_576363.docx,16_2023_QH15,UNKNOWN,NaN
171,18_VBHN-VPQH_m_699248.docx,18_VBHN-VPQH,UNKNOWN,NaN
235,308_TB-VPCP_m_494560.docx,308_TB-VPCP,UNKNOWN,NaN
260,36-NQ_TW_m_519030.docx,36-NQ_TW,UNKNOWN,NaN
285,41_2024_QH15_m_622824.docx,41_2024_QH15,UNKNOWN,NaN
298,45_2019_QH14_432162.docx,45_2019_QH14_432162,UNKNOWN,NaN
310,495_BYT-MT_m_496440.docx,495_BYT-MT,UNKNOWN,NaN


- 101/VBHN-VPQH, 11/VBHN-BTC, 125_VBHN-VPQH, 18_VBHN-VPQH, 68_VBHN-VPQH, 91_VBHN-VPQH_712297: Integrated document
- 16/2023/QH15, 41_2024_QH15, 45/2019/QH14: Law
- 308_TB-VPCP: Announcement
- 36-NQ_TW: Resolution
- 495_BYT-MT: remove	
- 55-NQ/TW: Resolution

In [22]:
# Manually verified types for documents not identified from their headings
MANUAL_TYPE_OVERRIDES = {
    "101_VBHN-VPQH_m_694902.docx": "Integrated Document",
    "11_VBHN-BTC_m_609280.docx": "Integrated Document",
    "125_VBHN-VPQH_m_682303.docx": "Integrated Document",
    "18_VBHN-VPQH_m_699248.docx": "Integrated Document",
    "68_VBHN-VPQH_m_716092.docx": "Integrated Document",
    "91_VBHN-VPQH_712297.docx": "Integrated Document",

    "16_2023_QH15_m_576363.docx": "Law",
    "41_2024_QH15_m_622824.docx": "Law",
    "45_2019_QH14_432162.docx": "Law",

    "308_TB-VPCP_m_494560.docx": "Announcement",
    "36-NQ_TW_m_519030.docx": "Resolution",
    "55-NQ_TW_m_519058.docx": "Resolution"
}

# Files that should not be included in the regulatory corpus
EXCLUDED_FILES = {
    "495_BYT-MT_m_496440.docx"
}

In [23]:
# Replace UNKNOWN values with the manually verified document types
document_types["observed_type"] = document_types.apply(
    lambda row: MANUAL_TYPE_OVERRIDES.get(row["filename"],
                                          row["observed_type"]),
    axis=1)

# Mark excluded files without deleting the original documents
document_types["include"] = ~document_types["filename"].isin(EXCLUDED_FILES)

# Create the final document list used by the RAG pipeline
included_document_types = document_types[document_types["include"]].copy()

In [24]:
# Summarize the corrected document types
type_summary = (included_document_types.groupby("observed_type")
                .agg(document_count=("filename", "count"),
                     example_files=("filename",
                                    lambda files: list(files[:5])))
                .reset_index()
                .sort_values("document_count", ascending=False)
                .reset_index(drop=True))

display(type_summary)

,observed_type,document_count,example_files
0,Circular,158,"[01_2021_TT-BXD_m_488637.docx, 01_2022_TT-BNNP..."
1,Decree,102,"[01_2019_ND-CP_m_420475.docx, 02_VBHN-BKHCN_m_..."
2,Law,67,"[02_2026_QH16_m_706740.docx, 04_2017_QH14_m_35..."
3,Decision,56,"[01_2022_QD-TTg_m_567258.docx, 04_2017_QD-TTg_..."
4,Resolution,12,"[138_NQ-CP_m_544532.docx, 189_2025_QH15_m_6536..."
5,Integrated Document,6,"[101_VBHN-VPQH_m_694902.docx, 11_VBHN-BTC_m_60..."
6,Announcement,1,[308_TB-VPCP_m_494560.docx]
7,Directive,1,[03_CT-NHNN_m_348086.docx]
8,Notification,1,[30_TB-VPCP_m_507657.docx]
9,Order,1,[11_2001_L-CTN_m_71892.docx]


In [25]:
# Check the final dataset after manual corrections and exclusion
print("Original documents: ", len(document_types))
print("Included documents: ", len(included_document_types))
print("Excluded documents: ", (~document_types["include"]).sum())
print("Unknown documents:  ",
      (included_document_types["observed_type"] == "UNKNOWN").sum())

Original documents:  407
Included documents:  406
Excluded documents:  1
Unknown documents:   0


In [26]:
# ── Boilerplate — non-informative lines to strip ──────────────
BOILERPLATE = {
    "SOCIALIST REPUBLIC OF VIETNAM",
    "Independence – Freedom - Happiness", "Independence - Freedom - Happiness",
    "Independence – Freedom – Happiness", "--------", "-------",
    "---------------", "----------", "OF THE POLITICAL BUREAU",
}

# Standardize boilerplate values for case-insensitive comparison
BOILERPLATE = {
    line.upper() for line in BOILERPLATE
}

In [27]:
# ── Vietnam legal document hierarchy (1=highest, 99=unknown) ──
# Based on: Vietnam's System of Legal Normative Documents chart
# Cross-referenced with actual document types observed in corpus
VIETNAM_DOC_HIERARCHY = {
    # ── National Assembly level ───────────────────────────────
    "Constitution"           : 1,
    "Code"                   : 2,   # Bộ luật
    "Law"                    : 3,   # Luật — includes Integrated/Consolidated docs (VBHN)
    "Integrated Document"    : 3,   # VBHN — merged laws, same authority level as Law
    "Resolution"             : 4,   # Nghị quyết (National Assembly / Central Committee)
    "Ordinance"              : 5,   # Pháp lệnh — observed in corpus
    "Order"                  : 6,   # Lệnh (President) — observed in corpus
    # ── Government / Prime Minister level ─────────────────────
    "Decision"               : 7,   # Quyết định — President / PM / State Auditor General
    "Decree"                 : 8,   # Nghị định → ND-CP
    "Prime Minister Decision": 9,   # Quyết định Thủ tướng → QD-TTg
    # ── Ministerial / Judicial level ──────────────────────────
    "Circular"               : 10,  # Thông tư → TT-BCT, TT-BTC etc.
    "Joint Circular"         : 10,  # Thông tư liên tịch — same level as Circular
    "Directive"              : 11,  # Chỉ thị — observed in corpus
    # ── Notifications / Announcements ─────────────────────────
    "Notification"           : 12,  # Thông báo → TB-VPCP — observed in corpus
    "Announcement"           : 12,  # Similar to Notification — not in chart, treat equally
    # ── Local level ───────────────────────────────────────────
    "Provincial Resolution"  : 13,
    "Provincial Decision"    : 14,
    "District Resolution"    : 15,
    "District Decision"      : 16,
    "Commune Resolution"     : 17,
    "Commune Decision"       : 18,
}

In [28]:
# Map filename codes to document types as a fallback when the type
# cannot be identified from the document heading
DOC_TYPE_MAP = {
    "ND-CP"      : "Decree",                "NQ-CP"     : "Resolution",
    "NQ-TW"      : "Resolution",            "QD-TTg"    : "Decision",
    "QD-UBND"    : "Decision",              "QD-CTN"    : "Decision",
    "TT-BCT"     : "Circular",              "TT-BTC"    : "Circular",
    "TT-BNNPTNT" : "Circular",              "TT-BNNMT"  : "Circular",
    "TT-BTNMT"   : "Circular",              "TT-BYT"    : "Circular",
    "TT-BKHDT"   : "Circular",              "TT-BKHCN"  : "Circular",
    "TT-BXD"     : "Circular",              "TT-BGTVT"  : "Circular",
    "TT-BLDTBXH" : "Circular",              "TT-BCA"    : "Circular",
    "TT-BQP"     : "Circular",              "TT-BTTTT"  : "Circular",
    "TT-BXD-MT"  : "Circular",              "TT-BKHCNMT": "Circular",
    "L-CTN"      : "Law",                   "QH15"      : "Law",
    "VBHN-VPQH"  : "Integrated Document",   "TB-VPCP"   : "Announcement",
    "TW"         : "Resolution",
}


In [29]:
# ── ESG taxonomy verified against ESG_law_dataset_-_ESG_Categories.pdf ──
ESG_TAXONOMY = {
    ("Environment", "Air Emissions")                                         : ["GHG Emissions", "GHG Policies", "Non-GHG Air Emissions"],
    ("Environment", "Biodiversity")                                          : ["Biodiversity", "Forests"],
    ("Environment", "Environmental management system, reporting and fines."): ["Climate Risk Management", "Environmental Fines", "Environmental Management System", "Environmental Policy", "Environmental Reporting"],
    ("Environment", "Sustainable Product Development")                       : ["Green Products", "Green Buildings", "Resource Efficiency"],
    ("Environment", "Sustainable Production")                                : ["Green Products", "Green Buildings", "Resource Efficiency", "Energy", "Hazardous Waste", "Packaging", "Toxic Spills", "Waste", "Water"],
    ("Environment", "Electromagnetic Fields")                                : ["Electromagnetic Fields"],
    ("Environment", "GMOs")                                                  : ["GMOS"],
    ("Environment", "Ozone-depleting gases")                                 : ["Ozone-Depleting Gases"],
    ("Social", "Clinical Trials")                                            : ["Clinical Trials"],
    ("Social", "Collective Bargaining and Remuneration")                     : ["Collective Bargaining", "Remuneration"],
    ("Social", "Community and Society")                                      : ["Community and Society", "Public Health", "Philanthropy"],
    ("Social", "Customer Relationship")                                      : ["Customer Relationship"],
    ("Social", "HIV Programs")                                               : ["HIV Programs"],
    ("Social", "Human Rights")                                               : ["Child Labor", "Diversity", "Human Rights", "Indigenous Rights", "Access to Basic Services", "Access to Healthcare", "Animal Welfare"],
    ("Social", "Labor Practices and Employee Development")                   : ["Employee Development", "Employee Turnover", "Health and Safety", "Labor Practices", "Unions"],
    ("Social", "Product Safety")                                             : ["Product Safety"],
    ("Social", "Responsible Marketing")                                      : ["Responsible Marketing"],
    ("Social", "Supply Chain")                                               : ["Supply Chain"],
    ("Governance", "Corporate Governance, Strategy and Sustainability")      : ["Strategy", "Board", "Board Diversity", "ESG Incentives", "Corporate Governance", "Reporting Quality", "Chairperson-CEO Separation", "Governance"],
    ("Governance", "Business Ethics")                                        : ["Business Ethics", "Anti-competitive Practices", "Corruption", "Privacy and IT"],
    ("Governance", "Financial Inclusion")                                    : ["Financial Inclusion"],
    ("Governance", "Global Compact Membership")                              : ["Global Compact Membership"],
    ("Governance", "Shareholders")                                           : ["Shareholders"],
    ("Governance", "Site Closure")                                           : ["Site Closure"],
    ("Governance", "Taxes")                                                  : ["Taxes"],
    ("Governance", "Lobbying")                                               : ["Lobbying"],
    ("Governance", "Systemic Risk")                                          : ["Systemic Risk"],
}

In [30]:
# Use one known document to test each function separately
TEST_FILE = CONVERTED_DOCX_FOLDER / "01_2019_ND-CP_m_420475.docx"

print("File exists:", TEST_FILE.exists())
print("Filename:", TEST_FILE.name)

File exists: True
Filename: 01_2019_ND-CP_m_420475.docx


In [31]:
# define
# Preserve every Domain → Group → Category path, including duplicate categories
ESG_CLASSIFICATION_PATHS = [
    (category.lower(), domain, group, category)
    for (domain, group), categories in ESG_TAXONOMY.items()
    for category in categories]

In [32]:
# test
# Check categories that occur under more than one ESG group
green_product_paths = [
    path for path in ESG_CLASSIFICATION_PATHS
    if path[0] == "green products"]

print(green_product_paths)

[('green products', 'Environment', 'Sustainable Product Development', 'Green Products'), ('green products', 'Environment', 'Sustainable Production', 'Green Products')]


In [33]:
# define 
# Extract the identifying code section from a document filename
def get_doc_code(filename: str) -> str:

    # Remove the website ID and file extension
    filename_core = Path(filename).stem.split("_m_")[0]

    # Remove a trailing website ID when "_m_" is absent
    filename_core = re.sub(r"_\d{5,}$", "", filename_core)

    return filename_core

In [34]:
# test 
# Test filename-code extraction with different formats
test_filenames = [
    "01_2019_ND-CP_m_420475.docx",
    "36-NQ_TW_m_519030.docx",
    "45_2019_QH14_432162.docx",
    "91_VBHN-VPQH_712297.docx"
]

for filename in test_filenames:
    print(filename, "→", get_doc_code(filename))

01_2019_ND-CP_m_420475.docx → 01_2019_ND-CP
36-NQ_TW_m_519030.docx → 36-NQ_TW
45_2019_QH14_432162.docx → 45_2019_QH14
91_VBHN-VPQH_712297.docx → 91_VBHN-VPQH


In [35]:
# Extract the type from document content before using filename fallback.
def get_document_type(paragraphs: list,
                      filename: str) -> tuple:

    known_types = {
        "CONSTITUTION": "Constitution",
        "CODE": "Code",
        "LAW": "Law",
        "RESOLUTION": "Resolution",
        "ORDINANCE": "Ordinance",
        "ORDER": "Order",
        "DECREE": "Decree",
        "DECISION": "Decision",
        "CIRCULAR": "Circular",
        "JOINT CIRCULAR": "Joint Circular",
        "DIRECTIVE": "Directive",
        "NOTIFICATION": "Notification",
        "ANNOUNCEMENT": "Announcement"
}

    # Search the document heading first
    for paragraph in paragraphs[:20]:
        normalized_paragraph = " ".join(paragraph.upper().split()).strip(" .:-")

        if normalized_paragraph in known_types:
            return known_types[normalized_paragraph], "document_heading"

    # Use the filename map only when no heading is found
    filename_code = get_doc_code(filename).upper().replace("_", "-")

    for code in sorted(DOC_TYPE_MAP, key=len, reverse=True):
        if code.upper() in filename_code:
            return DOC_TYPE_MAP[code], f"filename:{code}"

    return "Unknown", "not_found"

In [36]:
# Extract paragraphs and irregular tables in document order.
def get_paragraphs(docx_path: Path) -> list:
    document = Document(docx_path)
    texts = []

    for block in document.iter_inner_content():

        # Extract a normal paragraph
        if isinstance(block, Paragraph):
            text = " ".join(block.text.split())

            if text:
                texts.append(text)

        # Extract table XML without relying on the table grid
        else:
            for row_xml in block._tbl.tr_lst:
                cells = []

                for cell_xml in row_xml.tc_lst:
                    cell_text = "".join(
                        node.text or ""
                        for node in cell_xml.iter()
                        if node.tag == qn("w:t")
                    )

                    cells.append(" ".join(cell_text.split()))

                if any(cells):
                    texts.append("TABLE ROW | " + " | ".join(cells))

    return texts

In [37]:
# Test content-first document-type extraction
test_paragraphs = get_paragraphs(TEST_FILE)

document_type, type_source = get_document_type(
                                 paragraphs=test_paragraphs,
                                 filename=TEST_FILE.name)

print("Document type:", document_type)
print("Extraction source:", type_source)

Document type: Decree
Extraction source: document_heading


In [38]:
# Inspect the extracted paragraphs
test_paragraphs = get_paragraphs(TEST_FILE)

print("Paragraph count:", len(test_paragraphs))

for paragraph in test_paragraphs[:10]:
    print(paragraph)

Paragraph count: 178
TABLE ROW | THE GOVERNMENT-------- | SOCIALIST REPUBLIC OF VIETNAMIndependence – Freedom - Happiness---------------
TABLE ROW | No: 01/2019/ND-CP | Hanoi, January 01 2019
DECREE
ON FOREST RANGERS AND FOREST PROTECTION FORCES OF FOREST OWNERS
Pursuant to Law on the Government organization dated June 19 2015;
Pursuant to Law on Forestry dated November 15 2017;
At the proposal of the Minister of Natural Resources and Environment;
The Government promulgates a Decree on forest rangers and forest protection forces of forest owners.
Chapter I.
GENERAL PROVISIONS


In [39]:
# Reconstruct the official number from the filename
def get_official_number(filename: str) -> str:

    filename_core = get_doc_code(filename)

    # Convert filename separators into the official-number format
    return filename_core.replace("_", "/")

In [40]:
# Test reconstruction of official document numbers
for filename in test_filenames:
    print(filename, "→", get_official_number(filename))

01_2019_ND-CP_m_420475.docx → 01/2019/ND-CP
36-NQ_TW_m_519030.docx → 36-NQ/TW
45_2019_QH14_432162.docx → 45/2019/QH14
91_VBHN-VPQH_712297.docx → 91/VBHN-VPQH


In [41]:
# Find the first Chapter, Section or Article heading.
def find_body_start(paragraphs: list) -> int:
    body_pattern = re.compile(
        r"^(Chapter|Section|Article)\s+[A-Z0-9IVX]+[.:\s]",
        flags=re.IGNORECASE)

    for index, paragraph in enumerate(paragraphs):
        if body_pattern.match(paragraph):
            return index

    # Keep the complete document when no body heading is found
    return 0

In [42]:
# test
# Check where the substantive legal body begins
body_start = find_body_start(test_paragraphs)

print("Body starts at index:", body_start)
print("First body line:", test_paragraphs[body_start])

Body starts at index: 8
First body line: Chapter I.


In [43]:
# Extract the line following the document-type heading.
def get_title(paragraphs: list) -> str:

    document_keywords = {
        "CONSTITUTION", "CODE", "LAW", "RESOLUTION", "ORDINANCE",
        "ORDER", "DECREE", "DECISION", "CIRCULAR", "JOINT CIRCULAR",
        "DIRECTIVE", "NOTIFICATION", "ANNOUNCEMENT"
    }

    for index, paragraph in enumerate(paragraphs[:20]):
        normalized_paragraph = paragraph.upper().strip(" .:-")

        if normalized_paragraph in document_keywords:
            if index + 1 < len(paragraphs):
                return paragraphs[index + 1]

    return ""

In [44]:
# Test title extraction
document_title = get_title(test_paragraphs)

print("Title:", document_title)

Title: ON FOREST RANGERS AND FOREST PROTECTION FORCES OF FOREST OWNERS


In [45]:
# Extract the issuer from a promulgation sentence when available.

def get_issuing_body(paragraphs: list) -> str:

    issuing_pattern = re.compile(
        r"^(The Government|The Prime Minister|"
        r"Minister of [A-Za-z ,&-]+?|"
        r"Ministers? of [A-Za-z ,&-]+?)"
        r"\s+(?:hereby\s+)?(?:promulgates?|issues?|has promulgated)",
        flags=re.IGNORECASE)

    for paragraph in paragraphs[:20]:
        match = issuing_pattern.match(paragraph.strip())

        if match:
            return match.group(1).strip()

    return ""

In [46]:
# Test issuing-body extraction
issuing_body = get_issuing_body(test_paragraphs)

print("Issuing body:", issuing_body or "Not found")

Issuing body: The Government


In [47]:
# Extract a standalone issue date from the document header.
def get_issue_date(paragraphs: list) -> str:

    date_pattern = re.compile(
        r"^(?:Hanoi,\s*)?"
        r"([A-Z][a-z]+\s+\d{1,2},?\s+\d{4})$")

    for paragraph in paragraphs[:15]:
        match = date_pattern.match(paragraph.strip())

        if match:
            return match.group(1)

    return ""

In [48]:
# Blank is acceptable when the translated document omits its issue date
issue_date = get_issue_date(test_paragraphs)

print("Issue date:", issue_date or "Not found in document header")

Issue date: Not found in document header


In [49]:
# Extract an effective-date statement from the document.
def get_effective_date(paragraphs: list) -> str:

    effective_pattern = re.compile(
        r"(?:takes effect|comes into force|enters into force)"
        r"(?:\s+on|\s+from)?\s+(.+?)(?:\.|$)",
        flags=re.IGNORECASE)

    for paragraph in paragraphs:
        match = effective_pattern.search(paragraph)

        if match:
            return match.group(1).strip()

    return ""

In [50]:
# Test effective-date extraction
effective_date = get_effective_date(test_paragraphs)

print("Effective date:", effective_date or "Not found")

Effective date: February 15 2019


In [51]:
# Extract an Article by its heading, regardless of Article number.
def get_article_content(paragraphs: list,
                        heading_keywords: list) -> str:

    next_heading = re.compile(
        r"^(Article|Chapter|Section)\s+",
        flags=re.IGNORECASE)

    for index, paragraph in enumerate(paragraphs):
        is_article = re.match(r"^Article\s+\d+", paragraph,
                              flags=re.IGNORECASE)

        has_keyword = any(keyword.lower() in paragraph.lower()
                          for keyword in heading_keywords)

        if is_article and has_keyword:
            content = []

            # Collect content until the next Article, Chapter or Section
            for next_paragraph in paragraphs[index + 1:]:
                if next_heading.match(next_paragraph):
                    break

                content.append(next_paragraph)

            return " ".join(content)

    return ""

# Extract scope regardless of its Article number.
def get_scope(paragraphs: list) -> str:

    return get_article_content(
               paragraphs=paragraphs,
               heading_keywords=["scope", "governing scope"])

# Extract regulated entities regardless of their Article number.
def get_regulated_entities(paragraphs: list) -> str:

    return get_article_content(
               paragraphs=paragraphs,
               heading_keywords=["regulated entities",
                                 "subjects of application"])

In [52]:
# Test scope and regulated-entity extraction
print("Scope:")
print(get_scope(test_paragraphs))

print("\nRegulated entities:")
print(get_regulated_entities(test_paragraphs))

Scope:
This Decree stipulates the tasks, power, organization, equipment for operating and benefits for forest rangers and tasks, power, equipment for operating and benefits for forest protection forces of forest owners.

Regulated entities:
This Decree applies to forest rangers, forest protection forces of forest owners and agencies, organizations, individuals related to the activities thereof.


In [53]:
def classify_esg(body_text: str) -> tuple:
    """Suggest ESG labels using the approved taxonomy categories."""

    normalized_text = body_text.lower()
    matched_paths = []

    for keyword, domain, group, category in ESG_CLASSIFICATION_PATHS:
        if keyword in normalized_text:
            matched_paths.append({
                "domain": domain,
                "group": group,
                "category": category,
                "keyword": keyword
            })

    domains = sorted({match["domain"] for match in matched_paths})
    groups = sorted({match["group"] for match in matched_paths})
    categories = sorted({match["category"] for match in matched_paths})

    return domains, groups, categories, matched_paths

In [54]:
# Test ESG suggestions and inspect the evidence
test_text = "\n".join(test_paragraphs)

domains, groups, categories, matched_paths = classify_esg(test_text)

print("Domains:", domains)
print("Groups:", groups)
print("Categories:", categories)
display(pd.DataFrame(matched_paths))

Domains: ['Environment', 'Governance', 'Social']
Groups: ['Biodiversity', 'Corporate Governance, Strategy and Sustainability', 'Human Rights', 'Sustainable Production']
Categories: ['Biodiversity', 'Board', 'Diversity', 'Forests', 'Water']


,domain,group,category,keyword
0,Environment,Biodiversity,Biodiversity,biodiversity
1,Environment,Biodiversity,Forests,forests
2,Environment,Sustainable Production,Water,water
3,Social,Human Rights,Diversity,diversity
4,Governance,"Corporate Governance, Strategy and Sustainability",Board,board


In [55]:
# Clean extracted text while preserving substantive legal content.
def clean_body(paragraphs: list,
               body_start: int = 0) -> str:

    cleaned_paragraphs = []

    for paragraph in paragraphs[body_start:]:
        # Remove BOM and normalize whitespace
        cleaned_paragraph = paragraph.replace("\ufeff", "")
        cleaned_paragraph = " ".join(cleaned_paragraph.split()).strip()

        if not cleaned_paragraph:
            continue

        # Remove only verified boilerplate
        if cleaned_paragraph.upper() in BOILERPLATE:
            continue

        # Remove decorative separator lines
        if re.fullmatch(r"[-–—]{5,}", cleaned_paragraph):
            continue

        # Remove the translation-provider footer
        if "THƯ VIỆN PHÁP LUẬT" in cleaned_paragraph.upper():
            continue

        cleaned_paragraphs.append(cleaned_paragraph)

    return "\n".join(cleaned_paragraphs)

In [56]:
# Clean the document while preserving its title and preamble
cleaned_text = clean_body(paragraphs=test_paragraphs,
                          body_start=0)

print(cleaned_text[:2000])
print("\nWord count:", len(cleaned_text.split()))

TABLE ROW | THE GOVERNMENT-------- | SOCIALIST REPUBLIC OF VIETNAMIndependence – Freedom - Happiness---------------
TABLE ROW | No: 01/2019/ND-CP | Hanoi, January 01 2019
DECREE
ON FOREST RANGERS AND FOREST PROTECTION FORCES OF FOREST OWNERS
Pursuant to Law on the Government organization dated June 19 2015;
Pursuant to Law on Forestry dated November 15 2017;
At the proposal of the Minister of Natural Resources and Environment;
The Government promulgates a Decree on forest rangers and forest protection forces of forest owners.
Chapter I.
GENERAL PROVISIONS
Article 1. Scope
This Decree stipulates the tasks, power, organization, equipment for operating and benefits for forest rangers and tasks, power, equipment for operating and benefits for forest protection forces of forest owners.
Article 2. Regulated entities
This Decree applies to forest rangers, forest protection forces of forest owners and agencies, organizations, individuals related to the activities thereof.
Chapter II.
TASKS, PO

##### **Main extraction loop**

In [57]:
# Main loop extraction 
# Extract cleaned text and metadata from all included documents.
def extract_all(docx_folder: Path, cleaned_text_folder: Path,
                log_folder: Path, document_type_lookup: dict) -> list:

    # Create the required output folders
    cleaned_text_folder.mkdir(parents=True, exist_ok=True)
    log_folder.mkdir(parents=True, exist_ok=True)

    # Include subfolders and exclude manually rejected files
    files = sorted(path for path in docx_folder.rglob("*.docx")
                   if path.name not in EXCLUDED_FILES)

    registry = []

    print(f"Extracting {len(files)} documents...\n")

    for number, path in enumerate(files, start=1):
        try:
            # Extract paragraphs and table rows
            paragraphs = get_paragraphs(path)

            if not paragraphs:
                registry.append({
                    "doc_id": path.stem,
                    "source_filename": path.name,
                    "status": "empty",
                    "error": None
                })

                print(f"[{number}/{len(files)}] EMPTY | {path.name}")
                continue

            # Use the manually verified corpus type when available
            document_type = document_type_lookup.get(path.name)

            if document_type:
                type_source = (
                    "manual_override"
                    if path.name in MANUAL_TYPE_OVERRIDES
                    else "document_heading"
                )
            else:
                document_type, type_source = get_document_type(
                                                 paragraphs=paragraphs,
                                                 filename=path.name)

            # Preserve the title and legal preamble in the cleaned text
            cleaned_text = clean_body(paragraphs=paragraphs,
                                      body_start=0)

            # Generate ESG suggestions for later manual verification
            domains, groups, categories, matched_paths = classify_esg(
                                                             cleaned_text)

            # Save one cleaned text file per document
            cleaned_path = cleaned_text_folder / f"{path.stem}.txt"
            cleaned_path.write_text(cleaned_text, encoding="utf-8")

            # Store document metadata
            metadata = {
                "doc_id": path.stem,
                "official_number": get_official_number(path.name),
                "document_type": document_type,
                "document_type_source": type_source,
                "document_type_code": get_doc_code(path.name),
                "title": get_title(paragraphs),
                "issuing_body": get_issuing_body(paragraphs),
                "issue_date": get_issue_date(paragraphs),
                "effective_date": get_effective_date(paragraphs),
                "scope": get_scope(paragraphs),
                "regulated_entities": get_regulated_entities(paragraphs),
                "esg_domains": domains,
                "esg_category_groups": groups,
                "esg_categories": categories,
                "esg_matches": matched_paths,
                "esg_review_status": "pending",
                "source_filename": path.name,
                "cleaned_path": str(cleaned_path),
                "word_count": len(cleaned_text.split()),
                "char_count": len(cleaned_text),
                "table_row_count": sum(
                    text.startswith("TABLE ROW |")
                    for text in paragraphs
                ),
                "status": "ok",
                "error": None
            }

            registry.append(metadata)

            esg_label = "/".join(domains) if domains else "unclassified"

            print(f"[{number}/{len(files)}] {document_type:20} | "
                  f"{metadata['word_count']:6} words | "
                  f"{metadata['table_row_count']:4} table rows | "
                  f"{esg_label} | {path.name}")

        except Exception as error:
            registry.append({
                "doc_id": path.stem,
                "source_filename": path.name,
                "status": "error",
                "error": str(error)
            })

            print(f"[{number}/{len(files)}] ERROR | "
                  f"{path.name} | {error}")

    # Save the complete metadata registry
    registry_path = log_folder / "corpus_registry.json"
    registry_path.write_text(
        json.dumps(registry, indent=2, ensure_ascii=False),
        encoding="utf-8")

    successful = sum(item["status"] == "ok"
                     for item in registry)

    empty = sum(item["status"] == "empty"
                for item in registry)

    errors = sum(item["status"] == "error"
                 for item in registry)

    print("\n" + "=" * 55)
    print("Documents processed:", len(registry))
    print("Successful:         ", successful)
    print("Empty:              ", empty)
    print("Errors:             ", errors)
    print("Registry:           ", registry_path)
    print("=" * 55)

    return registry

In [58]:
# Use document types already checked and manually corrected
document_type_lookup = (
    included_document_types
    .set_index("filename")["observed_type"]
    .to_dict()
)

In [59]:
# Extract cleaned text and metadata from all included documents
# registry = extract_all(docx_folder=CONVERTED_DOCX_FOLDER,
                       # cleaned_text_folder=CLEANED_TEXT_FOLDER,
                       # log_folder=LOG_FOLDER,
                       # document_type_lookup=document_type_lookup)

In [ ]:
# Load completed extraction unless reprocessing is explicitly requested

REGISTRY_PATH = LOG_FOLDER / "extraction_registry.json"
FORCE_REEXTRACT = False

if REGISTRY_PATH.exists() and not FORCE_REEXTRACT:
    registry = json.loads(REGISTRY_PATH.read_text(encoding="utf-8"))
    missing_outputs = [item for item in registry if item.get("status") == "ok"
                       and (not item.get("cleaned_path") or not Path(item["cleaned_path"]).exists())]

    if not missing_outputs:
        print(f"Extraction already completed — loaded {len(registry)} records.")
        print("Registry:", REGISTRY_PATH)
    else:
        print(f"{len(missing_outputs)} cleaned files are missing — running extraction.")
        registry = extract_all(docx_folder=CONVERTED_DOCX_FOLDER,
                               cleaned_text_folder=CLEANED_TEXT_FOLDER,
                               log_folder=LOG_FOLDER,
                               document_type_lookup=document_type_lookup)
else:
    registry = extract_all(docx_folder=CONVERTED_DOCX_FOLDER,
                           cleaned_text_folder=CLEANED_TEXT_FOLDER,
                           log_folder=LOG_FOLDER,
                           document_type_lookup=document_type_lookup)

registry_df = pd.DataFrame(registry)

Extracting 406 documents...

[1/406] Decree               |   3727 words |    3 table rows | Environment/Governance/Social | 01_2019_ND-CP_m_420475.docx
[2/406] Circular             |  21348 words |  286 table rows | Environment/Social | 01_2021_TT-BXD_m_488637.docx
[3/406] Decision             |    793 words |   31 table rows | Environment | 01_2022_QD-TTg_m_567258.docx
[4/406] Circular             |  16278 words |  584 table rows | Environment/Governance/Social | 01_2022_TT-BNNPTNT_m_511704.docx
[5/406] Circular             |   4745 words |    3 table rows | Environment/Social | 01_2022_TT-BTNMT_m_528824.docx
[6/406] Circular             |   4179 words |    3 table rows | Environment | 01_2024_TT-BKHCN_600112.docx
[7/406] Circular             |   8561 words |  139 table rows | Environment | 01_2025_TT-BNNMT_m_659004.docx
[8/406] Circular             |  11588 words |    3 table rows | Environment | 01_2026_TT-BNNMT_m_693953.docx
[9/406] Circular             |   1491 words |   14 table

In [63]:
# Check source files, cleaned files and registry records
docx_files = list(CONVERTED_DOCX_FOLDER.rglob("*.docx"))
cleaned_files = list(CLEANED_TEXT_FOLDER.rglob("*.txt"))

print("Converted .docx files:", len(docx_files))
print("Cleaned .txt files:   ", len(cleaned_files))
print("Registry records:     ", len(registry))
print("Successful records:   ", sum(item.get("status") == "ok" for item in registry))
print("Empty records:        ", sum(item.get("status") == "empty" for item in registry))
print("Error records:        ", sum(item.get("status") == "error" for item in registry))

Converted .docx files: 407
Cleaned .txt files:    406
Registry records:      406
Successful records:    406
Empty records:         0
Error records:         0


#### **Validate final results**

In [64]:
# Convert the registry into a table for validation
registry_df = pd.DataFrame(registry)

print("Expected documents:", len(included_document_types))
print("Registry documents:", len(registry_df))
print("Successful:",
      (registry_df["status"] == "ok").sum())
print("Errors:",
      (registry_df["status"] == "error").sum())

Expected documents: 406
Registry documents: 406
Successful: 406
Errors: 0


In [65]:
# Display the document that failed during extraction
error_files = registry_df[
    registry_df["status"] == "error"]

display(error_files[[
    "source_filename",
    "doc_id",
    "error"]])

,source_filename,doc_id,error


In [66]:
# Test the repaired table extraction on the failed document
ERROR_TEST_FILE = (
    CONVERTED_DOCX_FOLDER
    / "70_2023_ND-CP_m_580154.docx"
)

error_test_content = get_paragraphs(ERROR_TEST_FILE)

print("Extracted blocks:", len(error_test_content))
print("Table rows:",
      sum(text.startswith("TABLE ROW |")
          for text in error_test_content))
print("Extraction successful:", len(error_test_content) > 0)

Extracted blocks: 318
Table rows: 104
Extraction successful: True


In [67]:
# Reprocess only the previously failed document
path = ERROR_TEST_FILE
paragraphs = get_paragraphs(path)

document_type = document_type_lookup.get(path.name, "Unknown")
cleaned_text = clean_body(paragraphs=paragraphs,
                          body_start=0)

domains, groups, categories, matches = classify_esg(cleaned_text)

# Save its cleaned text
cleaned_path = CLEANED_TEXT_FOLDER / f"{path.stem}.txt"
cleaned_path.write_text(cleaned_text, encoding="utf-8")

# Recreate its metadata
repaired_metadata = {"doc_id": path.stem,
                     "official_number": get_official_number(path.name),
                     "document_type": document_type,
                     "document_type_source": "document_heading",
                     "document_type_code": get_doc_code(path.name),
                     "title": get_title(paragraphs),
                     "issuing_body": get_issuing_body(paragraphs),
                     "issue_date": get_issue_date(paragraphs),
                     "effective_date": get_effective_date(paragraphs),
                     "scope": get_scope(paragraphs),
                     "regulated_entities": get_regulated_entities(paragraphs),
                     "esg_domains": domains,
                     "esg_category_groups": groups,
                     "esg_categories": categories,
                     "esg_matches": matches,
                     "esg_review_status": "pending",
                     "source_filename": path.name,
                     "cleaned_path": str(cleaned_path),
                     "word_count": len(cleaned_text.split()),
                     "char_count": len(cleaned_text),
                     "table_row_count": sum( text.startswith("TABLE ROW |") for text in paragraphs),
                     "status": "ok",
                     "error": None
}

In [68]:
# Find and replace the failed registry row
error_index = registry_df.index[
    registry_df["source_filename"] == path.name][0]

for column, value in repaired_metadata.items():
    registry_df.at[error_index, column] = value

In [69]:
# Confirm that all 406 documents now succeeded
print("Registry documents:", len(registry_df))
print("Successful:", (registry_df["status"] == "ok").sum())
print("Errors:", (registry_df["status"] == "error").sum())

Registry documents: 406
Successful: 406
Errors: 0


In [70]:
# Save the corrected complete registry
registry = registry_df.to_dict(orient="records")

registry_path = LOG_FOLDER / "corpus_registry.json"
registry_path.write_text(
    json.dumps(registry, indent=2, ensure_ascii=False),
    encoding="utf-8")

print("Updated registry:", registry_path)

Updated registry: /Users/tanggiee/Desktop/RAG_AI/esg_rag_project/outputs/logs/corpus_registry.json


In [73]:
# Find successful documents with no ESG domain assigned "unclassified"
unclassified_files = registry_df[(registry_df["status"] == "ok")
                                 & registry_df["esg_domains"].apply(
                                     lambda value: not isinstance(value, list) or len(value) == 0)].copy()

print("Unclassified files:", len(unclassified_files))

# Display the complete unclassified list
with pd.option_context("display.max_rows", None,
                       "display.max_colwidth", 120):
    display(unclassified_files[[
        "source_filename",
        "official_number",
        "document_type",
        "title"
    ]])

Unclassified files: 59


,source_filename,official_number,document_type,title
10,02_2025_TT-BCT_m_648178.docx,02/2025/TT-BCT,Circular,ON PROTECTION OF ELECTRICITY WORKS AND SAFETY IN ELECTRICAL SECTOR
19,03_2025_TT-BLDTBXH_m_646590.docx,03/2025/TT-BLDTBXH,Circular,ON WORK CLASSIFICATION STANDARDS BASED ON WORKING CONDITIONS
27,04_2021_TT-BCT_m_482504.docx,04/2021/TT-BCT,Circular,ON WORKING HOURS AND RESTING HOURS OF WORKERS WORKING IN UNDERGROUND MINES
29,04_2023_TT-BNNPTNT_m_576528.docx,04/2023/TT-BNNPTNT,Circular,PROMULGATING THE LIST OF LIVING THINGS SUBJECT TO PLANT QUARANTINE IN VIETNAM
35,05_2015_TT-BNNPTNT_m_284118.docx,05/2015/TT-BNNPTNT,Circular,REGULATIONS ON SEQUENCE AND PROCEDURES FOR ISSUANCE OF PRACTISING CERTIFICATE FOR TREATMENT OF PLANT QUARANTINE SUBJ...
41,06_2025_TT-BNNMT_m_671280.docx,06/2025/TT-BNNMT,Circular,ON NATIONAL TECHNICAL REGULATION ON EMISSIONS OF IN-USE AUTOMOBILES
44,07_2023_TT-BTC_m_556248.docx,07/2023/TT-BTC,Circular,"PRESCRIBING FEES FOR APPRAISAL OF ENVIRONMENTAL REMEDIATION AND IMPROVEMENT PLANS BY CENTRAL-LEVEL AUTHORITIES, COLL..."
47,07_2026_TT-BNNMT_703640.docx,07/2026/TT-BNNMT,Circular,"ELABORATING CERTAIN ARTICLES OF THE LAW ON PLANT PROTECTION AND QUARANTINE, LAW ON CROP CULTIVATION AND AMENDMENTS T..."
57,09_2009_TT-BKHCN_m_199506.docx,09/2009/TT-BKHCN,Circular,PROVIDING GUIDANCE ON THE REQUIREMENTS AND PROCEDURE FOR DESIGNATION OF CONFORMITY ASSESSMENT BODIES
63,1009_QD-BCT_m_656831.docx,1009/QD-BCT,Decision,APPROVING THE GENERATION PRICE BRACKET FOR COAL POWER PLANTS IN 2025


In [74]:
# Return the unclassified filenames as a Python list
unclassified_filename_list = unclassified_files["source_filename"].tolist()

print(unclassified_filename_list)

['02_2025_TT-BCT_m_648178.docx', '03_2025_TT-BLDTBXH_m_646590.docx', '04_2021_TT-BCT_m_482504.docx', '04_2023_TT-BNNPTNT_m_576528.docx', '05_2015_TT-BNNPTNT_m_284118.docx', '06_2025_TT-BNNMT_m_671280.docx', '07_2023_TT-BTC_m_556248.docx', '07_2026_TT-BNNMT_703640.docx', '09_2009_TT-BKHCN_m_199506.docx', '1009_QD-BCT_m_656831.docx', '10_2014_UBTVQH13_m_267874.docx', '10_2021_TT-BKHDT_m_502913.docx', '112_2021_TT-BTC_m_529021.docx', '115_2014_TT-BTC_m_283734.docx', '116_2014_ND-CP_m_261978.docx', '116_2018_ND-CP_m_397485.docx', '11_2001_L-CTN_m_71892.docx', '12_2022_TT-BCT_m_582930.docx', '144_2024_ND-CP_m_647300.docx', '14_2023_TT-BNNPTNT_m_594179.docx', '15_2022_TT-BNNPTNT_m_536367.docx', '15_2024_TT-BNNPTNT_m_635285.docx', '17_2022_TT-NHNN_m_548392.docx', '17_2026_TT-BNNMT_m_703611.docx', '18_2025_ND-CP_m_650598.docx', '199_2025_ND-CP_m_664759.docx', '19_2009_TT-BKHCN_m_96067.docx', '203_2025_QH15_m_661982.docx', '20_2009_TT-BXD_m_358651.docx', '20_2023_QH15_m_577937.docx', '20_2023_T

#### **Handle unclassfied files**


In [75]:
# Temporary terms for classifying currently unclassified documents
# Every key must correspond to an existing ESG_TAXONOMY path
TEMP_ESG_KEYWORD_ALIASES = {
    # Environment
    ("Environment", "Air Emissions", "Non-GHG Air Emissions"): [
        "emission standard",
        "vehicle exhaust",
        "motorcycle emission",
        "motor vehicle emission",
        "vehicle emission",
        "exhaust emission"
    ],

    ("Environment", "Biodiversity", "Biodiversity"): [
        "plant quarantine",
        "animal quarantine",
        "plant protection",
        "quarantine pest",
        "invasive species",
        "aquatic resource",
        "fisheries resource"
    ],

    ("Environment",
     "Environmental management system, reporting and fines.",
     "Climate Risk Management"): [
        "natural disaster management",
        "disaster prevention",
        "disaster response",
        "emergency response"
    ],

    ("Environment",
     "Environmental management system, reporting and fines.",
     "Environmental Management System"): [
        "environmental remediation",
        "environmental improvement plan",
        "environmental impact assessment",
        "environmental permit",
        "environmental inspection"
    ],

    ("Environment",
     "Environmental management system, reporting and fines.",
     "Environmental Policy"): [
        "environmental technical regulation",
        "environmental protection regulation",
        "environmental protection requirement"
    ],

    ("Environment", "Sustainable Production", "Resource Efficiency"): [
        "mineral resource",
        "mineral reserve",
        "mineral extraction",
        "mineral operation",
        "natural resource exploitation"
    ],

    ("Environment", "Sustainable Production", "Energy"): [
        "electricity trade",
        "electricity supply",
        "power plant",
        "coal power",
        "electricity generation",
        "power generation"
    ],

    ("Environment", "Sustainable Production", "Waste"): [
        "sea dumping",
        "waste-related emergency",
        "waste incident",
        "waste response"
    ],

    ("Environment", "Sustainable Production", "Water"): [
        "marine pollution",
        "aquaculture environment",
        "sea water quality",
        "water discharge"
    ],

    # Social
    ("Social", "Human Rights", "Animal Welfare"): [
        "animal husbandry",
        "livestock breed",
        "animal breed",
        "livestock management"
    ],

    ("Social", "Product Safety", "Product Safety"): [
        "quality of goods",
        "conformity assessment",
        "medical supplies",
        "medical device",
        "product quality inspection",
        "technical conformity"
    ],

    ("Social", "Supply Chain", "Supply Chain"): [
        "origin traceability",
        "certificate of origin",
        "product origin",
        "supply origin"
    ],

    ("Social", "Community and Society", "Public Health"): [
        "plant quarantine",
        "animal quarantine",
        "disease prevention",
        "public health protection"
    ],

    # Governance
    ("Governance", "Taxes", "Taxes"): [
        "value-added tax",
        "export tariff",
        "import tariff",
        "special excise duty",
        "tax administration",
        "tax exemption"
    ],

    ("Governance", "Systemic Risk", "Systemic Risk"): [
        "financial risk",
        "insurance policy",
        "agricultural insurance",
        "risk management",
        "emergency management"
    ],

    ("Governance", "Business Ethics", "Business Ethics"): [
        "state inspection",
        "market inspection",
        "regulatory compliance",
        "administrative violation"
    ]
}

In [76]:
# Build all valid Domain → Group → Category paths
valid_taxonomy_paths = {
    (domain, group, category)
    for (domain, group), categories in ESG_TAXONOMY.items()
    for category in categories
}

# Find temporary paths that do not exist in ESG_TAXONOMY
invalid_alias_paths = (
    set(TEMP_ESG_KEYWORD_ALIASES)
    - valid_taxonomy_paths
)

print("Temporary alias paths:", len(TEMP_ESG_KEYWORD_ALIASES))
print("Invalid paths:", invalid_alias_paths)

Temporary alias paths: 16
Invalid paths: set()


In [77]:
# Classify text using official categories and temporary aliases
# temporary functions to work on unclassfied documents 
def classify_esg_temporary(body_text: str) -> tuple:

    normalized_text = body_text.lower()
    matches = []

    for (domain, group), categories in ESG_TAXONOMY.items():
        for category in categories:
            taxonomy_path = (domain, group, category)

            # Combine the official category name with temporary aliases
            keywords = [category.lower()]
            keywords += TEMP_ESG_KEYWORD_ALIASES.get(taxonomy_path, [])

            matched_keywords = [
                keyword
                for keyword in keywords
                if re.search(rf"\b{re.escape(keyword.lower())}\b",
                             normalized_text)]

            if matched_keywords:
                matches.append({
                    "domain": domain,
                    "group": group,
                    "category": category,
                    "matched_keywords": sorted(set(matched_keywords))})

    # Remove duplicate labels while preserving valid taxonomy results
    domains = sorted({match["domain"] for match in matches})
    groups = sorted({match["group"] for match in matches})
    categories = sorted({match["category"] for match in matches})

    return domains, groups, categories, matches

In [78]:
# Select successful documents with no ESG domain
unclassified_indexes = registry_df[
    (registry_df["status"] == "ok")
    & registry_df["esg_domains"].apply(
        lambda value: not isinstance(value, list) or len(value) == 0)
].index

temporary_results = []

for index in unclassified_indexes:
    cleaned_path = Path(registry_df.at[index, "cleaned_path"])
    cleaned_text = cleaned_path.read_text(encoding="utf-8")

    domains, groups, categories, matches = (
        classify_esg_temporary(cleaned_text)
    )

    # Update only documents that receive a temporary classification
    if domains:
        registry_df.at[index, "esg_domains"] = domains
        registry_df.at[index, "esg_category_groups"] = groups
        registry_df.at[index, "esg_categories"] = categories
        registry_df.at[index, "esg_matches"] = matches
        registry_df.at[index, "esg_review_status"] = (
            "pending_temporary_alias_review"
        )

    temporary_results.append({
        "source_filename": registry_df.at[index, "source_filename"],
        "esg_domains": domains,
        "esg_category_groups": groups,
        "esg_categories": categories,
        "matched_keywords": [
            keyword
            for match in matches
            for keyword in match["matched_keywords"]
        ]
    })

temporary_results_df = pd.DataFrame(temporary_results)

In [79]:
# Show newly classified documents first
display(temporary_results_df[temporary_results_df["esg_domains"].apply(bool)])

,source_filename,esg_domains,esg_category_groups,esg_categories,matched_keywords
0,02_2025_TT-BCT_m_648178.docx,"[Environment, Social]","[Product Safety, Sustainable Production]","[Energy, Product Safety]","[electricity generation, power generation, pow..."
3,04_2023_TT-BNNPTNT_m_576528.docx,"[Environment, Social]","[Biodiversity, Community and Society]","[Biodiversity, Public Health]","[plant protection, plant quarantine, plant qua..."
4,05_2015_TT-BNNPTNT_m_284118.docx,"[Environment, Social]","[Biodiversity, Community and Society]","[Biodiversity, Public Health]","[plant protection, plant quarantine, plant qua..."
6,07_2023_TT-BTC_m_556248.docx,"[Environment, Governance]","[Environmental management system, reporting an...","[Environmental Management System, Taxes]","[environmental remediation, tax administration]"
7,07_2026_TT-BNNMT_703640.docx,[Environment],[Biodiversity],[Biodiversity],[plant protection]
8,09_2009_TT-BKHCN_m_199506.docx,[Social],[Product Safety],[Product Safety],[conformity assessment]
9,1009_QD-BCT_m_656831.docx,[Environment],[Sustainable Production],[Energy],"[coal power, electricity generation]"
11,10_2021_TT-BKHDT_m_502913.docx,[Environment],"[Environmental management system, reporting an...",[Climate Risk Management],[disaster prevention]
12,112_2021_TT-BTC_m_529021.docx,"[Environment, Governance]","[Environmental management system, reporting an...","[Environmental Management System, Taxes]","[environmental remediation, tax administration]"
13,115_2014_TT-BTC_m_283734.docx,[Governance],[Systemic Risk],[Systemic Risk],[insurance policy]


In [80]:
# Select documents that remain unclassified
still_unclassified = temporary_results_df[
    ~temporary_results_df["esg_domains"].apply(bool)].copy()

# Add document code and title from the main registry
still_unclassified = still_unclassified.merge(registry_df[["source_filename", "official_number", "title"]],
                                              on="source_filename", how="left")

# Rename columns for easier reading
still_unclassified = still_unclassified.rename(columns={
    "source_filename": "file_name",
    "official_number": "file_code"})

# Show the complete table without hiding rows or shortening titles
with pd.option_context("display.max_rows", None,
                       "display.max_colwidth", None):
    display(
        still_unclassified[[
            "file_code",
            "file_name",
            "title"
        ]].reset_index(drop=True)
    )

# Count all documents that remain unclassified
print("Total unclassified documents:", len(still_unclassified))

,file_code,file_name,title
0,03/2025/TT-BLDTBXH,03_2025_TT-BLDTBXH_m_646590.docx,ON WORK CLASSIFICATION STANDARDS BASED ON WORKING CONDITIONS
1,04/2021/TT-BCT,04_2021_TT-BCT_m_482504.docx,ON WORKING HOURS AND RESTING HOURS OF WORKERS WORKING IN UNDERGROUND MINES
2,06/2025/TT-BNNMT,06_2025_TT-BNNMT_m_671280.docx,ON NATIONAL TECHNICAL REGULATION ON EMISSIONS OF IN-USE AUTOMOBILES
3,10/2014/UBTVQH13,10_2014_UBTVQH13_m_267874.docx,ENVIRONMENTAL POLICE FORCES
4,12/2022/TT-BCT,12_2022_TT-BCT_m_582930.docx,"WORKING HOURS, REST PERIODS FOR WORKERS OPERATING, MAINTAINING, REPAIRING GAS DISTRIBUTION PIPELINES AND GAS WORKS"
5,17/2026/TT-BNNMT,17_2026_TT-BNNMT_m_703611.docx,"ON AMENDMENTS TO SOME ARTICLES OF CIRCULAR NO. 25/2018/TT-BNNPTNT DATED NOVEMBER 15, 2018 OF MINISTER OF AGRICULTURE AND ENVIRONMENT ON PROCEDURES FOR RISK ASSESSMENT OF AND LICENSE FOR IMPORT OF LIVE AQUATIC ANIMALS AND PLANTS AMENDED BY CIRCULAR NO. 01/2022/TT-BNNPTNT DATED JANUARY 18, 2022 ON AMENDMENTS TO CIRCULARS IN AQUACULTURE"
6,19/2009/TT-BKHCN,19_2009_TT-BKHCN_m_96067.docx,ON QUALITY CONTROL MEASURES FOR PRODUCTS AND GOODS SUBJECT TO INCREASED MANAGEMENT BEFORE MARKET CIRCULATION
7,203/2025/QH15,203_2025_QH15_m_661982.docx,AMENDMENTS TO CERTAIN ARTICLES OF THE CONSTITUTION OF THE SOCIALIST REPUBLIC OF VIETNAM
8,20/2009/TT-BXD,20_2009_TT-BXD_m_358651.docx,"ON AMENDMENTS TO THE CIRCULAR NO. 20/2005/TT-BXD DATED DECEMBER 20, 2005 BY THE MINISTRY OF CONSTRUCTION ON GUIDELINES FOR URBAN FORESTRY"
9,20/2023/QH15,20_2023_QH15_m_577937.docx,ELECTRONIC TRANSACTIONS


Total unclassified documents: 22


KEEP: 
- 06/2025/TT-BNNMT: ON NATIONAL TECHNICAL REGULATION ON EMISSIONS OF IN-USE AUTOMOBILES
- 10/2014/UBTVQH13: ENVIRONMENTAL POLICE FORCES
- 29/2026/ND-CP: ON DOMESTIC CARBON EXCHANGE
- 42/2025/TT-BKHCN: APPLICATION OF STANDARDS AND TECHNICAL REGULATIONS TO DATA CENTERS

In [81]:
# Manually verified ESG labels for the four retained documents
MANUAL_ESG_OVERRIDES = {
    "06/2025/TT-BNNMT": {
        "esg_domains": ["Environment"],
        "esg_category_groups": ["Air Emissions"],
        "esg_categories": ["Non-GHG Air Emissions"]
    },

    "10/2014/UBTVQH13": {
        "esg_domains": ["Environment"],
        "esg_category_groups": [
            "Environmental management system, reporting and fines."
        ],
        "esg_categories": ["Environmental Policy"]
    },

    "29/2026/ND-CP": {
        "esg_domains": ["Environment"],
        "esg_category_groups": ["Air Emissions"],
        "esg_categories": ["GHG Policies"]
    },

    "42/2025/TT-BKHCN": {
        "esg_domains": ["Environment"],
        "esg_category_groups": ["Sustainable Production"],
        "esg_categories": ["Resource Efficiency"]
    }
}

In [82]:
# Apply manual ESG labels and retain all unclassified documents for later review
for document_code, labels in MANUAL_ESG_OVERRIDES.items():
    matching_indexes = registry_df.index[registry_df["official_number"].eq(document_code)]
    for index in matching_indexes:
        registry_df.at[index, "esg_domains"] = labels["esg_domains"]
        registry_df.at[index, "esg_category_groups"] = labels["esg_category_groups"]
        registry_df.at[index, "esg_categories"] = labels["esg_categories"]
        registry_df.at[index, "esg_matches"] = []
        registry_df.at[index, "esg_review_status"] = "manually_classified"

# Mark remaining unclassified documents without removing them
unclassified_mask = registry_df["esg_domains"].apply(lambda value: not isinstance(value, list) or len(value) == 0)
registry_df.loc[unclassified_mask, "esg_review_status"] = "needs_manual_review"

manual_documents = registry_df[registry_df["official_number"].isin(MANUAL_ESG_OVERRIDES)]
unclassified_documents = registry_df[unclassified_mask].copy()

display(manual_documents[["official_number", "title", "esg_domains", "esg_category_groups", "esg_categories", "esg_review_status"]])
display(unclassified_documents[["official_number", "title", "esg_review_status"]].reset_index(drop=True))

print("Manually classified:", len(manual_documents))
print("Needs manual review:", len(unclassified_documents))
print("Registry documents:", len(registry_df))

,official_number,title,esg_domains,esg_category_groups,esg_categories,esg_review_status
41,06/2025/TT-BNNMT,ON NATIONAL TECHNICAL REGULATION ON EMISSIONS ...,[Environment],[Air Emissions],[Non-GHG Air Emissions],manually_classified
76,10/2014/UBTVQH13,ENVIRONMENTAL POLICE FORCES,[Environment],"[Environmental management system, reporting an...",[Environmental Policy],manually_classified
233,29/2026/ND-CP,ON DOMESTIC CARBON EXCHANGE,[Environment],[Air Emissions],[GHG Policies],manually_classified
289,42/2025/TT-BKHCN,APPLICATION OF STANDARDS AND TECHNICAL REGULAT...,[Environment],[Sustainable Production],[Resource Efficiency],manually_classified


,official_number,title,esg_review_status
0,03/2025/TT-BLDTBXH,ON WORK CLASSIFICATION STANDARDS BASED ON WORK...,needs_manual_review
1,04/2021/TT-BCT,ON WORKING HOURS AND RESTING HOURS OF WORKERS ...,needs_manual_review
2,12/2022/TT-BCT,"WORKING HOURS, REST PERIODS FOR WORKERS OPERAT...",needs_manual_review
3,17/2026/TT-BNNMT,ON AMENDMENTS TO SOME ARTICLES OF CIRCULAR NO....,needs_manual_review
4,19/2009/TT-BKHCN,ON QUALITY CONTROL MEASURES FOR PRODUCTS AND G...,needs_manual_review
5,203/2025/QH15,AMENDMENTS TO CERTAIN ARTICLES OF THE CONSTITU...,needs_manual_review
6,20/2009/TT-BXD,ON AMENDMENTS TO THE CIRCULAR NO. 20/2005/TT-B...,needs_manual_review
7,20/2023/QH15,ELECTRONIC TRANSACTIONS,needs_manual_review
8,20/2023/TT-BCT,WORKING HOURS AND REST PERIODS OF OFFSHORE PET...,needs_manual_review
9,25/2022/TT-BTTTT,"REGULATIONS ON DETERMINING IMPORTED MATERIALS,...",needs_manual_review


Manually classified: 4
Needs manual review: 18
Registry documents: 406


In [83]:
# check for duplicated files 

In [84]:
# check for duplicated files & repeated official numbers 
official_number_counts = (
    registry_df.groupby("official_number")
    .size()
    .reset_index(name="document_count"))

duplicate_numbers = official_number_counts[
    official_number_counts["document_count"] > 1
].sort_values("document_count", ascending=False)

print("Repeated official numbers:", len(duplicate_numbers))
print("Documents involved:", duplicate_numbers["document_count"].sum())

display(duplicate_numbers)

Repeated official numbers: 3
Documents involved: 6


,official_number,document_count
112,13/2021/TT-BNNPTNT,2
192,21/QD-BCT,2
360,72/2020/QH14,2


In [85]:
# Show every file associated with a repeated official number
duplicate_files = registry_df[
    registry_df["official_number"].isin(
        duplicate_numbers["official_number"]
    )
][[
    "official_number",
    "doc_id",
    "source_filename",
    "title"
]].sort_values([
    "official_number",
    "source_filename"
])

display(duplicate_files)

,official_number,doc_id,source_filename,title
122,13/2021/TT-BNNPTNT,13_2021_TT-BNNPTNT_m_495964 (1),13_2021_TT-BNNPTNT_m_495964 (1).docx,PROVIDING FOR COMPLIANCE WITH REQUIREMENTS FOR...
123,13/2021/TT-BNNPTNT,13_2021_TT-BNNPTNT_m_495964,13_2021_TT-BNNPTNT_m_495964.docx,PROVIDING FOR COMPLIANCE WITH REQUIREMENTS FOR...
195,21/QD-BCT,21_QD-BCT_m_550028 (1),21_QD-BCT_m_550028 (1).docx,PROMULGATION OF THE TRANSITIONAL FRAMEWORK FOR...
196,21/QD-BCT,21_QD-BCT_m_550028,21_QD-BCT_m_550028.docx,PROMULGATION OF THE TRANSITIONAL FRAMEWORK FOR...
362,72/2020/QH14,72_2020_QH14_463512,72_2020_QH14_463512.docx,ON ENVIRONMENTAL PROTECTION
363,72/2020/QH14,72_2020_QH14_m_463512,72_2020_QH14_m_463512.docx,ON ENVIRONMENTAL PROTECTION


In [86]:
# Prefer original filenames over copies ending in "(1)"
registry_df = registry_df.sort_values("source_filename",
                                      key=lambda names: names.str.contains(r"\(\d+\)", regex=True))

# Keep one document for each official number
duplicate_mask = registry_df.duplicated("official_number", keep="first")
removed_duplicates = registry_df[duplicate_mask].copy()

registry_df = registry_df[~duplicate_mask].reset_index(drop=True)

display(removed_duplicates[["official_number", "source_filename", "title"]])

print("Duplicates removed:", len(removed_duplicates))
print("Documents remaining:", len(registry_df))

,official_number,source_filename,title
362,72/2020/QH14,72_2020_QH14_463512.docx,ON ENVIRONMENTAL PROTECTION
195,21/QD-BCT,21_QD-BCT_m_550028 (1).docx,PROMULGATION OF THE TRANSITIONAL FRAMEWORK FOR...
122,13/2021/TT-BNNPTNT,13_2021_TT-BNNPTNT_m_495964 (1).docx,PROVIDING FOR COMPLIANCE WITH REQUIREMENTS FOR...


Duplicates removed: 3
Documents remaining: 403


In [87]:
# Save the corrected complete registry
registry = registry_df.to_dict(orient="records")

registry_path = LOG_FOLDER / "corpus_registry.json"
registry_path.write_text(
    json.dumps(registry, indent=2, ensure_ascii=False),
    encoding="utf-8")

print("Saved documents:", len(registry))
print("Updated registry:", registry_path)

Saved documents: 403
Updated registry: /Users/tanggiee/Desktop/RAG_AI/esg_rag_project/outputs/logs/corpus_registry.json
